# Mutual k-NN Stimulus Visualizer

For each EEG model (maximum size) x best-matching vision model, this notebook:

1. Computes per-window mKNN overlap at the best (EEG layer, vision layer) pair.
2. Selects the **top-5 stimulus windows** with the highest mutual neighbor count.
3. For each anchor window, displays three rows of animated GIFs:
   - **Shared** (blue) -- anchor + windows in **both** the EEG k-NN and the vision k-NN.
   - **EEG-only** (red) -- windows the EEG considers similar but the vision model does not.
   - **Vision-only** (green) -- windows the vision model considers similar but EEG does not.

**EEG k-NN**: subject-averaged embeddings (consensus across 6 subjects).  
**Vision models**: VideoMAE (base / large) and DINOv2 (small / base / large / giant).  
**Videos**: downloaded from Fudan-fMRI/CineBrain on first run (~2.6 GB, cached).  
**k = 5** mutual nearest neighbors.

A second analysis block repeats everything with **temporal exclusion (+-10 windows)**,
removing temporally adjacent clips from the neighbor pool to prevent spurious
matches driven by shot continuity rather than semantic content.

In [ ]:
from pathlib import Path
import os, io, base64, tarfile, functools
import numpy as np
import cv2
from PIL import Image as PILImage
from IPython.display import HTML, display as ipy_display
from sklearn.neighbors import NearestNeighbors
from huggingface_hub import hf_hub_download

# ── HuggingFace repos ─────────────────────────────────────────────────────────
EMBED_REPO = 'nitrox639/platonic-embeddings'
VIDEO_REPO = 'Fudan-fMRI/CineBrain'

for _tp in (Path('tokens/hf_token.txt'), Path('hf_token.txt'),
            Path.home() / 'hf_token.txt',
            Path('..') / 'tokens' / 'hf_token.txt'):
    if _tp.exists():
        os.environ['HF_TOKEN'] = _tp.read_text().strip()
        break
HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'No HF token found — put it in tokens/hf_token.txt at the repo root.'

def fetch_embed(p):
    return hf_hub_download(EMBED_REPO, p, repo_type='dataset', token=HF_TOKEN)

def load_npz(p):
    return np.load(fetch_embed(p), allow_pickle=True)

# ── EEG model registry ────────────────────────────────────────────────────────
EEG = {
    'femba':       {'sizes': ['tiny', 'base', 'large'],  'family': 'femba_luna',
                    'window_s': 5,
                    'fname': lambda s: f'eeg/femba/femba_{s}_tusl_layerwise.npz'},
    'luna':        {'sizes': ['base', 'large', 'huge'],  'family': 'femba_luna',
                    'window_s': 5,
                    'fname': lambda s: f'eeg/luna/luna_{s}_layerwise.npz'},
    'neurolm':     {'sizes': ['b', 'l', 'xl'],           'family': 'neurolm',
                    'window_s': 8,
                    'fname': lambda s: f'eeg/neurolm/neurolm_{s}_layerwise.npz'},
    'steegformer': {'sizes': ['small', 'base', 'large'], 'family': 'steegformer',
                    'window_s': 6,
                    'fname': lambda s: f'eeg/steegformer/steegformer_{s}_layerwise.npz'},
    'reve':        {'sizes': ['base', 'large'],          'family': 'reve',
                    'window_s': 10,
                    'fname': lambda s: f'eeg/reve/reve_{s}_layerwise.npz'},
}
VIDEOMAE_SIZES = ['base', 'large']
DINOV2_SIZES   = ['small', 'base', 'large', 'giant']
DISPLAY = {'femba': 'FEMBA', 'luna': 'LUNA', 'neurolm': 'NeuroLM',
           'steegformer': 'STEEGFormer', 'reve': 'REVE'}

# ── CineBrain clip parameters ─────────────────────────────────────────────────
CLIP_SECONDS = 4.0
SOURCE_FPS   = 8
MAX_CLIP_IDX = 2699
SEASON_END_S = (MAX_CLIP_IDX + 1) * CLIP_SECONDS   # 10 800 s

# ── Analysis hyperparameters ──────────────────────────────────────────────────
K_MKNN      = 5    # k for mutual nearest neighbors
TOP_N       = 5    # top stimulus windows to visualize
GIF_SIZE    = 160  # pixel size for GIF display
EXCL_RADIUS = 10   # ±windows excluded in temporal-exclusion analysis

print('Setup complete.')

In [ ]:
# ── kNN / mKNN helpers ────────────────────────────────────────────────────────

def l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def knn_1d(z, k):
    """Cosine k-NN on L2-normalised (W, D) -> (W, k)."""
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine',
                          algorithm='brute').fit(z)
    _, idx = nn.kneighbors(z)
    return idx[:, 1:].astype(np.int32)

def knn_1d_excl(z, k, excl_radius=10):
    """
    Cosine k-NN excluding temporal neighbors within ±excl_radius.
    z must be L2-normalised. Uses full pairwise cosine via dot product.
    """
    W = z.shape[0]
    D = (1.0 - z @ z.T).astype(np.float32)   # cosine distance (W, W)
    rows = np.arange(W)
    D[np.abs(rows[:, None] - rows[None, :]) <= excl_radius] = np.inf
    return np.argsort(D, axis=1)[:, :k].astype(np.int32)

def mknn_1d(ia, ib):
    return float((ia[:, :, None] == ib[:, None, :]).any(-1).mean())

def mknn_per_window(ia, ib):
    return (ia[:, :, None] == ib[:, None, :]).any(-1).mean(axis=-1)

def canonical_starts(window_s):
    return np.arange(0.0, SEASON_END_S, float(window_s), dtype=np.float64)

# ── Per-layer k-NN builders ───────────────────────────────────────────────────

def precompute_knn_eeg_avg(emb, k, knn_fn=knn_1d):
    """EEG (n_layers, W, S, D) → subject-avg → (n_layers, W, k)."""
    emb_avg  = emb.mean(axis=2)
    n_layers, W, _ = emb_avg.shape
    idx = np.zeros((n_layers, W, k), dtype=np.int32)
    for l in range(n_layers):
        idx[l] = knn_fn(l2(emb_avg[l]), k)
        if (l + 1) % 4 == 0:
            print(f'    EEG layer {l+1}/{n_layers} done')
    return idx

def precompute_knn_layers(emb, k, knn_fn=knn_1d):
    """Vision (n_layers, W, D) → (n_layers, W, k)."""
    emb = np.asarray(emb)
    if emb.ndim == 2:
        emb = emb[None]
    n_layers, W, _ = emb.shape
    idx = np.zeros((n_layers, W, k), dtype=np.int32)
    for l in range(n_layers):
        idx[l] = knn_fn(l2(emb[l]), k)
    return idx

# ── Reusable analysis pipeline ────────────────────────────────────────────────

def run_analysis(vision_sizes, vpath_fn, vision_label,
                 eeg_knn_fn=None, vis_knn_fn=None):
    """
    For each EEG model (max size):
      1. Compute subject-averaged EEG k-NN.
      2. For each vision model size, compute vision k-NN and raw max mKNN.
      3. Pick the best vision size, find best (EEG layer × vision layer) pair.
      4. Rank windows by per-window mKNN; keep top-N.

    eeg_knn_fn / vis_knn_fn: callables (emb, k) -> (n_layers, W, k).
    Defaults to standard cosine k-NN (no temporal exclusion).
    """
    _eeg_knn = eeg_knn_fn or (lambda emb, k: precompute_knn_eeg_avg(emb, k))
    _vis_knn = vis_knn_fn or (lambda emb, k: precompute_knn_layers(emb, k))

    results = {}
    for model, cfg in EEG.items():
        eeg_size = cfg['sizes'][-1]
        fam      = cfg['family']
        window_s = cfg['window_s']
        starts_s = canonical_starts(window_s)
        W        = len(starts_s)

        print(f"\n{'='*62}")
        print(f"  {DISPLAY[model]}-{eeg_size}  |  {vision_label}  |  W={W}")

        eeg_emb = load_npz(cfg['fname'](eeg_size))['embeddings'].astype(np.float32)
        print(f"  EEG shape: {eeg_emb.shape}")
        nn_eeg  = _eeg_knn(eeg_emb, K_MKNN)
        LE      = nn_eeg.shape[0]
        print(f"  EEG k-NN: {nn_eeg.shape}")

        best_vs, best_score, best_nn_vis = None, -1.0, None
        for vs in vision_sizes:
            vemb  = load_npz(vpath_fn(vs, fam))['embeddings'].astype(np.float32)
            nn_v  = _vis_knn(vemb, K_MKNN)
            LV    = nn_v.shape[0]
            score = max(mknn_1d(nn_eeg[le], nn_v[lv])
                        for le in range(LE) for lv in range(LV))
            print(f'  {vision_label}-{vs} ({LV} layers): raw_mknn={score:.4f}')
            if score > best_score:
                best_score, best_vs, best_nn_vis = score, vs, nn_v

        LV    = best_nn_vis.shape[0]
        S_obs = np.array([[mknn_1d(nn_eeg[le], best_nn_vis[lv])
                           for lv in range(LV)] for le in range(LE)])
        le_star, lv_star = np.unravel_index(S_obs.argmax(), S_obs.shape)
        print(f'  ▶ best: {best_vs}  EEG-l{le_star} × Vis-l{lv_star}'
              f'  mKNN={S_obs[le_star, lv_star]:.4f}')

        per_window = mknn_per_window(nn_eeg[le_star], best_nn_vis[lv_star])
        top_idx    = np.argsort(per_window)[::-1][:TOP_N]
        print(f'  top-{TOP_N}: {per_window[top_idx].round(3).tolist()}')

        results[model] = dict(
            eeg_size     = eeg_size,
            vis_size     = best_vs,
            vision_label = vision_label,
            window_s     = window_s,
            le_star      = int(le_star),
            lv_star      = int(lv_star),
            raw_score    = float(best_score),
            per_window   = per_window,
            top_indices  = top_idx,
            starts_s     = starts_s,
            nn_eeg_best  = nn_eeg[le_star],        # (W, k)
            nn_vis_best  = best_nn_vis[lv_star],   # (W, k)
        )
    return results

print('kNN helpers + run_analysis ready.')

In [ ]:
# ── Video helpers ─────────────────────────────────────────────────────────────

CB_ROOT    = Path('.').resolve().parent   # CineBrain root (notebook is in notebooks/)
CLIP_CACHE = CB_ROOT / '.clip_cache'
CLIP_MARKER = CLIP_CACHE / '.extracted'

_CLIP_INDEX: dict[int, Path] = {}

def _build_clip_index(clips_dir: Path):
    """Scan clips_dir for *.mp4 files and index them by clip number."""
    global _CLIP_INDEX
    _CLIP_INDEX.clear()
    for p in clips_dir.rglob('*.mp4'):
        digits = ''.join(c for c in p.stem if c.isdigit())
        if digits:
            _CLIP_INDEX[int(digits)] = p
    print(f'  Indexed {len(_CLIP_INDEX)} clips '
          f'(range {min(_CLIP_INDEX)}..{max(_CLIP_INDEX)})')

def ensure_clips():
    """Make sure video clips are available locally, downloading if needed."""
    if _CLIP_INDEX:
        return   # already built
    if CLIP_MARKER.exists():
        print(f'Using cached clips at {CLIP_CACHE}')
        _build_clip_index(CLIP_CACHE)
        return
    # Download videos.tar from HuggingFace
    print('Downloading videos.tar from HuggingFace (~2.6 GB, cached after first run)...')
    tar_path = Path(hf_hub_download(
        repo_id=VIDEO_REPO, filename='videos.tar',
        repo_type='dataset', token=HF_TOKEN))
    print(f'Extracting to {CLIP_CACHE} ...')
    CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, 'r') as tar:
        tar.extractall(CLIP_CACHE)
    CLIP_MARKER.touch()
    _build_clip_index(CLIP_CACHE)
    print('Clips ready.')

@functools.lru_cache(maxsize=128)
def _load_clip_frames(clip_idx: int) -> tuple:
    """Decode all frames of clip clip_idx as uint8 RGB; cached."""
    path = _CLIP_INDEX[clip_idx]
    cap  = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return tuple(frames)

def get_window_frames(start_s: float, window_s: float,
                      size: int = GIF_SIZE) -> list:
    """
    Extract all frames for [start_s, start_s + window_s) by stitching
    the necessary clips. Resize each frame to (size x size) pixels.
    """
    end_s      = start_s + window_s
    first_clip = int(start_s // CLIP_SECONDS)
    last_clip  = min(int(np.ceil(end_s / CLIP_SECONDS)) - 1, MAX_CLIP_IDX)

    frames = []
    for c in range(first_clip, last_clip + 1):
        clip_frames = _load_clip_frames(c)
        n       = len(clip_frames)
        clip_t0 = c * CLIP_SECONDS
        ov_t0   = max(start_s, clip_t0)
        ov_t1   = min(end_s,   clip_t0 + CLIP_SECONDS)
        f0 = max(0, int(np.floor((ov_t0 - clip_t0) / CLIP_SECONDS * n)))
        f1 = min(n, int(np.ceil ((ov_t1 - clip_t0) / CLIP_SECONDS * n)))
        for frame in clip_frames[f0:f1]:
            frames.append(cv2.resize(frame, (size, size),
                                     interpolation=cv2.INTER_LINEAR))
    return frames

def frames_to_gif_bytes(frames: list, fps: int = SOURCE_FPS) -> bytes:
    """Encode frame list (each H×W×3 uint8 RGB) as animated GIF bytes."""
    pil_frames = [PILImage.fromarray(f) for f in frames]
    buf = io.BytesIO()
    pil_frames[0].save(
        buf, format='GIF', save_all=True,
        append_images=pil_frames[1:],
        duration=int(1000 / fps),   # ms per frame
        loop=0,
        optimize=False)
    return buf.getvalue()

def gif_html(frames, fps=SOURCE_FPS, width=GIF_SIZE, border='none'):
    """Inline HTML <img> tag with the GIF embedded as base64."""
    b64 = base64.b64encode(frames_to_gif_bytes(frames, fps)).decode()
    return (f'<img src="data:image/gif;base64,{b64}" '
            f'width="{width}" '
            f'style="border:{border}; image-rendering:auto;"/>')

print('Video helpers ready. Call ensure_clips() before visualizing.')

In [ ]:
# ── VideoMAE analysis ─────────────────────────────────────────────────────────
results_videomae = run_analysis(
    VIDEOMAE_SIZES,
    lambda vs, fam: f'vision/videomae/videomae_{vs}__{fam}.npz',
    vision_label='VideoMAE',
)
print('\nVideoMAE analysis complete.')

In [ ]:
# ── DINOv2 analysis ───────────────────────────────────────────────────────────
results_dinov2 = run_analysis(
    DINOV2_SIZES,
    lambda vs, fam: f'vision/dinov2/dinov2_{vs}__{fam}.npz',
    vision_label='DINOv2',
)
print('\nDINOv2 analysis complete.')

In [ ]:
# ── Display function ──────────────────────────────────────────────────────────

def render_model(model: str, results: dict,
                 gif_size: int = GIF_SIZE, fps: int = SOURCE_FPS):
    """
    For each of the top-10 anchor windows, renders three rows:
      Row 1 – Shared     : anchor (gold) + windows in BOTH EEG and vision k-NN
      Row 2 – EEG-only   : windows in EEG k-NN but NOT vision k-NN
      Row 3 – Video-only : windows in vision k-NN but NOT EEG k-NN
    """
    res          = results[model]
    window_s     = res['window_s']
    starts_s     = res['starts_s']
    nn_eeg_best  = res['nn_eeg_best']
    nn_vis_best  = res['nn_vis_best']
    vis_label    = res['vision_label']

    def _card(ww, label, border='1px solid #ccc', bg='#fff'):
        t   = float(starts_s[ww])
        gif = gif_html(get_window_frames(t, float(window_s), size=gif_size),
                       fps=fps, width=gif_size, border=border)
        return (f'<div style="text-align:center; background:{bg}; '
                f'border-radius:4px; padding:4px; flex-shrink:0;">'
                f'{gif}<br/>'
                f'<span style="font-size:11px; font-family:monospace;">'
                f'{label}<br/>w={ww}&nbsp;{t:.1f}s '
                f'({int(t//60)}m{int(t%60):02d}s)</span></div>')

    def _row(title, color, windows, prefix=''):
        if not windows and not prefix:
            return ''
        cards = prefix + ''.join(_card(ww, 'neighbor') for ww in windows)
        return (f'<div style="margin:4px 0 10px 0;">'
                f'<div style="font-size:12px; font-weight:bold; color:{color}; '
                f'margin-bottom:4px;">{title}</div>'
                f'<div style="display:flex; flex-wrap:wrap; gap:6px;">'
                f'{cards}</div></div>')

    html = [f"""
    <div style="font-family:sans-serif; max-width:1400px;">
    <h2 style="border-bottom:2px solid #333; padding-bottom:6px;">
        {DISPLAY[model]}-{res['eeg_size']}
        &nbsp;&times;&nbsp; {vis_label}-{res['vis_size']}
    </h2>
    <p style="color:#555; font-size:13px;">
        Window = {window_s} s &nbsp;|&nbsp;
        EEG layer <b>{res['le_star']}</b> &times; Vision layer <b>{res['lv_star']}</b>
        &nbsp;|&nbsp; Best raw mKNN = <b>{res['raw_score']:.4f}</b>
        &nbsp;|&nbsp; k = {K_MKNN}
    </p>
    <p style="color:#888; font-size:12px;">
        <b style="color:#e6a817;">&#9632;</b> Anchor &nbsp;
        <b style="color:#2a7ae2;">&#9632;</b> Shared (EEG &cap; Vision) &nbsp;
        <b style="color:#c0392b;">&#9632;</b> EEG-only &nbsp;
        <b style="color:#27ae60;">&#9632;</b> Vision-only
    </p>"""]

    for rank, w in enumerate(res['top_indices']):
        t_anchor     = float(starts_s[w])
        overlap_frac = float(res['per_window'][w])

        eeg_set  = set(nn_eeg_best[w].tolist())
        vis_set  = set(nn_vis_best[w].tolist())
        shared   = sorted(eeg_set & vis_set)
        eeg_only = sorted(eeg_set - vis_set)
        vis_only = sorted(vis_set - eeg_set)

        anchor = _card(w, '<b>ANCHOR</b>',
                       border='3px solid #e6a817', bg='#fffbe6')

        html.append(f"""
        <div style="margin:20px 0; padding:12px; background:#f9f9f9;
                    border-radius:6px; border:1px solid #ddd;">
          <h3 style="margin:0 0 10px 0; font-size:14px;">
            #{rank+1} &nbsp; w=<b>{w}</b> &nbsp;|&nbsp;
            t = <b>{t_anchor:.1f} s</b> ({int(t_anchor//60)}m{int(t_anchor%60):02d}s)
            &nbsp;|&nbsp; overlap = <b>{overlap_frac:.2f}</b>
            &nbsp; ({len(shared)} shared / {len(eeg_only)} EEG-only / {len(vis_only)} Vision-only)
          </h3>
          {_row(f'Shared ({len(shared)}/{K_MKNN}) — EEG and {vis_label} agree',
                '#2a7ae2', shared, anchor)}
          {_row(f'EEG-only ({len(eeg_only)}/{K_MKNN}) — EEG says similar, {vis_label} does not',
                '#c0392b', eeg_only)}
          {_row(f'Vision-only ({len(vis_only)}/{K_MKNN}) — {vis_label} says similar, EEG does not',
                '#27ae60', vis_only)}
        </div>""")

    html.append('</div>')
    ipy_display(HTML(''.join(html)))

print('Display function ready.')

## Download / locate video clips

Run this cell once. If clips are already extracted locally they are reused immediately; otherwise videos.tar (~2.6 GB) is downloaded from HuggingFace and extracted to `.clip_cache/`.

In [ ]:
ensure_clips()

---
## EEG × VideoMAE — all models

Standard cosine k-NN (no temporal exclusion).

In [ ]:
for model in EEG:
    render_model(model, results_videomae)

---
## EEG × DINOv2 — all models

Standard cosine k-NN (no temporal exclusion).

In [ ]:
for model in EEG:
    render_model(model, results_dinov2)

---
## Temporal-exclusion analysis (±10 windows excluded)

Re-runs the full pipeline with `knn_1d_excl`, which removes the ±10 temporally
adjacent windows from each query's neighbor pool before finding k-NN.  
This prevents temporal autocorrelation (nearby clips in the show look similar
just because they're close in time) from inflating the shared-neighbor count.

In [ ]:
# ── Temporal-exclusion analysis ───────────────────────────────────────────────
_excl_eeg = lambda emb, k: precompute_knn_eeg_avg(
    emb, k, knn_fn=lambda z, k: knn_1d_excl(z, k, EXCL_RADIUS))
_excl_vis = lambda emb, k: precompute_knn_layers(
    emb, k, knn_fn=lambda z, k: knn_1d_excl(z, k, EXCL_RADIUS))

print(f'Temporal exclusion: ±{EXCL_RADIUS} windows\n')

results_videomae_excl = run_analysis(
    VIDEOMAE_SIZES,
    lambda vs, fam: f'vision/videomae/videomae_{vs}__{fam}.npz',
    vision_label=f'VideoMAE (excl ±{EXCL_RADIUS})',
    eeg_knn_fn=_excl_eeg,
    vis_knn_fn=_excl_vis,
)

results_dinov2_excl = run_analysis(
    DINOV2_SIZES,
    lambda vs, fam: f'vision/dinov2/dinov2_{vs}__{fam}.npz',
    vision_label=f'DINOv2 (excl ±{EXCL_RADIUS})',
    eeg_knn_fn=_excl_eeg,
    vis_knn_fn=_excl_vis,
)

print('\nTemporal-exclusion analysis complete.')

In [ ]:
for model in EEG:
    render_model(model, results_videomae_excl)

In [ ]:
for model in EEG:
    render_model(model, results_dinov2_excl)